# 1.1 - 1.2 Intro - Env setup, (LLM API key, model def)

In [ ]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# 1.3 RAG - Test the LLM directly with no context

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        model="llama-3.1-8b-instant", # modele chez groq equivalent
        # model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [ ]:
llm("Hey, what's up?")

'Not much. How can I help you today?'

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'm happy to chat with you. However, I'm a large language model, I don't have access to specific courses or have prior knowledge about you joining any course. 

Could you please provide more context or information about the course you're interested in? I'll do my best to guide you or provide general information about the course if available.


## Add context from FAQ

In [ ]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [ ]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
answer = llm(prompt)
print(answer)

I'd be happy to help you with your questions about the LLM Zoomcamp course.

You asked: "I just discovered the course. Can I join now?"

Answer: Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


OK, that's RAG (giving proper context to LLM)

## Retrieval plus generation

Goal: get something like that

In [1]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

# 1.4 Course FAQ Dataset 

Fetch directly FAQ (available as json format for ease of maintenance)

In [2]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [3]:
print(courses_raw)

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}]


Fetch all FAQs for all courses

In [4]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

Each entry has:

- `id` - unique identifier
- `course` - course slug (e.g., `machine-learning-zoomcamp`)
- `section` - which section of the course
- `question` - the FAQ question
- `answer` - the FAQ answer

Let's look at one:

In [5]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

## **Using this data**

In the RAG pipeline, dataset = knowledge base:

1. We **index** all the documents (the search step)
2. When a student asks a question, we **search the index**
3. The search returns the **most relevant** FAQ **entries**
4. We **give** those **entries to** the **LLM** as **context**
5. The **LLM generates** an **answer** based on the context

- The `question` and `answer` fields contain the text we'll search through.  
- The `course` field lets us filter by course. 
- For example, if a student asks about the data engineering course, we skip results from the ML course. 
- The `section` field helps with ranking - knowing which part of the course a question belongs to is useful context.

## **Data preparation**

- here: data **already prepared** (convenient JSON format). 
- A lot of this data **cleaned** (with the help of an LLM, handy use case on its own).
- In reality, **data preparation** is often the **most time-consuming part** of building a RAG system. 
  - scrape websites, 
  - parse PDFs, 
  - clean documents
  - chunk documents.

# 1.5 Search

At its core, every search engine does the same thing. It takes a **query**, **scores** every document for **similarity**, and returns the **top results**.

Formally, there is a similarity function:

In [7]:
# score = sim(query, document)

Main **difference** between search engines: **sim function**

2 ways:
- **text/lexical** search (module 1): 
  - `sim` counts the **number of words** the **query** and the **document** **share**. 
  - It looks at the **surface form**, the **actual words**, and **matches them** exactly.
- **vector/semantic** search (module 2): 
  - `sim` compares the **meaning** of the **query** and the **document**. 
  - Same function, **different similarity** measure.

Consider these two questions:

- "Can I still join the course after the start date?"
- "Is it possible to enroll late?"

They mean the same thing, but share almost no keywords. "Join" is not "enroll", "course" is absent, "start date" is not "late". A **text search** engine would **struggle** to match them, because it **only sees words**.

We'll see how vector search solves this later. For now, let's build text search with minsearch.

## Create an index with minsearch

Let's index the `documents`.

 Sending 1100 documents to the LLM would be **expensive** and **slow**. The model would get confused with **too much data**. Search finds the most relevant documents to send instead.

There are **many** search **libraries** you can use - Apache Lucene, Elasticsearch, Solr, and others (heavy). 

**minsearch** is a **simple in-memory** search engine (lightweight, runs anywhere Python runs, including Google Colab). It's a **toy implementation**, not production ready, but it **illustrates** **how search engines work** and it gives good results.

The concepts :
  - same as in Elasticsearch (which comes from Lucene): 
      - text fields, 
      - keyword fields, 
      - boosting, 
      - filtering. 

We'll index as **text** (they'll be tokenized and ranked) the fields:
  - `question` 
  - `section` 
  - `answer` 
, and as a **keyword** (for filtering) the field
  - `course`

The index **tokenizes text fields** and treats **keyword fields** as **exact strings**.

**Text fields** :
  - the **fields you search through**. 
  - When you type a query :
    - the search engine looks at these fields 
    - and **tokenizes** them. 
    - It **splits text into words**, 
    - **lowercases** them, 
    - **removes stop words**, and 
    - **ranks the results** by relevance. 
  So `question`, `section`, and `answer` are text fields.

**Keyword fields** :
  - for **exact matching** (filtering). 
  - **restrict the search space** to a particular** subset**. 


In [8]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

## ## **Trying a search**

Let's try a search with the question we used before:

In [9]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c